# Dist-SCAS Visualizations

This notebook contains plotting code (unexecuted) for the Dist-SCAS project. It shows how to generate publication-quality figures: quantile fan plots, CVaR training curves, IQR vs adaptive \lambda behavior, and AntMaze trajectory overlays.


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set(style="whitegrid", context="paper", font_scale=1.2)

In [ ]:
# Quantile fan plot for a fixed (s,a)
def plot_quantile_fan(quantile_matrix: np.ndarray, save_path: str):
    """quantile_matrix: [K, T] where K=N*M pooled quantiles across training checkpoints (T)
    or [T, K] depending on collection. This function plots the fan across quantile index.
    """
    q = np.array(quantile_matrix)
    if q.ndim == 2 and q.shape[0] < q.shape[1]:
        q = q.T
    # q: [T, K]
    median = np.median(q, axis=1)
    p10 = np.percentile(q, 10, axis=1)
    p90 = np.percentile(q, 90, axis=1)
    plt.figure(figsize=(6, 3.5))
    x = np.arange(q.shape[0])
    plt.fill_between(x, p10, p90, color="C0", alpha=0.2)
    plt.plot(x, median, color="C0", lw=2, label="median")
    plt.xlabel("Training step (or quantile index)")
    plt.ylabel("Return quantile")
    plt.title("Quantile fan (example)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)

In [ ]:
# IQR vs adaptive lambda scatter
def plot_iqr_lambda(iqr_vals: np.ndarray, lambda_vals: np.ndarray, save_path: str):
    plt.figure(figsize=(4.5, 3.5))
    plt.scatter(iqr_vals, lambda_vals, alpha=0.6)
    sns.regplot(iqr_vals, lambda_vals, scatter=False, lowess=True, color="C1")
    plt.xlabel("IQR (pooled quantiles)")
    plt.ylabel(r"\lambda(s,a)")
    plt.title("Adaptive \lambda vs IQR")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)

In [ ]:
# AntMaze trajectory overlay (dataset vs policy rollouts). Assumes 2D positions available in env info or obs.
def plot_antmaze_trajectories(
    dataset_states: np.ndarray,
    policy_trajs: list,
    map_img: np.ndarray = None,
    save_path: str = "antmaze_overlay.png",
):
    plt.figure(figsize=(6, 6))
    if map_img is not None:
        plt.imshow(map_img, origin="lower", cmap="gray", extent=[-1, 1, -1, 1])
    # dataset density (scatter subsample)
    ds = dataset_states[
        np.random.choice(
            len(dataset_states), size=min(2000, len(dataset_states)), replace=False
        )
    ]
    plt.scatter(ds[:, 0], ds[:, 1], s=4, alpha=0.3, label="dataset")
    for traj in policy_trajs:
        traj = np.array(traj)
        plt.plot(traj[:, 0], traj[:, 1], alpha=0.9)
    plt.legend()
    plt.title("AntMaze: dataset vs policy rollouts")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)

Usage notes:

- Run each plotting function after collecting the required arrays from training logs or model snapshots.
- Save figures under `paperAssignments/Assignments1-50/CA4/pictures/` and reference them in the LaTeX report.
- The notebook intentionally contains no executed outputs.
